# PDAP model configuration

This notebook shows how the runnable PDAP model is assembled from the
structured experiment config.

The flow is:

1. load the experiment YAML,
2. load the shared value-sample dataset,
3. apply the reversible normalization transform when requested,
4. resolve the activation registry entry, and
5. build the PDAP object through `PDAP.from_config(...)`.


In [1]:
from pathlib import Path

from omegaconf import OmegaConf

from src.PDAP import PDAP
from src.config.activations import get_activation
from src.data import load_value_samples, normalize_value_samples
from src.paths import DATA_DIR

cfg = OmegaConf.load(Path("../conf/experiment/activationsearch.yaml"))
cfg.model.activation = "softplus"
cfg.model.loss_weights = "h1"
cfg.model.gamma = 0.1
cfg.model.th = 0.5
cfg.model.c_init = 1.0
cfg.model.use_sphere = None
cfg.env = OmegaConf.create({"verbose": False})
data_path = DATA_DIR / cfg.data.path
data = load_value_samples(data_path)

normalizer = None
if cfg.data.normalize:
    data, normalizer = normalize_value_samples(data)

pdap = PDAP.from_config(cfg, data)

print(
    {
        "activation": cfg.model.activation,
        "use_sphere": get_activation(cfg.model.activation).use_sphere,
        "model": type(pdap.model).__name__,
        "insertion": pdap.insertion_kind,
        "train_samples": int(data["x"].shape[0]),
        "normalizer": normalizer.to_dict() if normalizer is not None else None,
    }
)


{'activation': 'softplus', 'use_sphere': False, 'model': 'SignedModel', 'insertion': 'profile', 'train_samples': 900, 'normalizer': {'x_scale': [3.0, 3.0], 'v_scale': 10.956660734061057}}


In [2]:
demo_data = {name: value[:64] for name, value in data.items()}
demo_pdap = PDAP.from_config(cfg, demo_data)

demo_training = OmegaConf.create(
    {
        "num_iterations": 1,
        "num_insertion": 1,
        "max_insert": 3,
        "threshold": 1e-5,
        "prune_merge_tol": 1e-3,
        "decorrelation": False,
    }
)

result = demo_pdap.fit_from_config(demo_training, verbose=False)
best_iteration = int(result["best_iteration"])
print(
    {
        "best_neurons": int(result["best_neurons"]),
        "val_h1": float(result["err_h1_val"][best_iteration]),
        "val_l2": float(result["err_l2_val"][best_iteration]),
    }
)


{'best_neurons': 1, 'val_h1': 0.7985474073838413, 'val_l2': 0.25323334866356895}


The full curated experiment runners use the same assembly path, but sweep
the configured parameter grids and write Run Records plus full fit-result
artifacts under `experiments/<name>/`.
